In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier

def build_features(df_input):
    frame = df_input.copy()
    
    # 1. DROP UNWANTED (Consistent with your previous list)
    drop_cols = [
        'customer_id', 'name', 'phone_number', 'address', 'branch_address',
        'branch_pin', 'branch_pin_code', 'freeze_date', 'unfreeze_date', 
        'account_frozen_flag', 'account_status', 'avg_balance_y', 'account_opening_date_y'
    ]
    frame = frame.drop(columns=[c for c in drop_cols if c in frame.columns], errors='ignore')

    # 2. DATE FEATURES
    date_cols = ["account_opening_date_x", "last_mobile_update_date", "last_kyc_date", 
                 "date_of_birth", "relationship_start_date", "last_transaction_date"]
    
    today = pd.Timestamp("2025-01-01")
    for col in date_cols:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors='coerce')
            # Create age features
            new_col_name = f"{col}_days"
            frame[new_col_name] = (today - frame[col]).dt.days
    
    frame = frame.drop(columns=[c for c in date_cols if c in frame.columns], errors='ignore')

    # 3. BINARY MAPPING
    binary_cols = ["nomination_flag","cheque_allowed","cheque_availed","kyc_compliant",
                   "rural_branch","pan_available","mobile_banking_flag","internet_banking_flag",
                   "atm_card_flag","demat_flag","credit_card_flag","fastag_flag",
                   "joint_account_flag","nri_flag"]
    for col in binary_cols:
        if col in frame.columns:
            frame[col] = frame[col].map({"Y": 1, "N": 0})

    # 4. LOG TRANSFORMS (Handling skewness)
    log_cols = ["credit_debit_ratio","ka_sum","quarterly_avg_balance","median_transaction_amount",
                "monthly_avg_balance","avg_balance_x","std_transaction_amount","avg_transaction_amount",
                "max_transaction_amount","max_credit_amount","max_debit_amount",
                "total_transaction_amount","total_credit_amount","total_debit_amount"]
    for col in log_cols:
        if col in frame.columns:
            frame[col] = np.log1p(np.abs(frame[col].fillna(0)))
            
    return frame

In [2]:
# Load Training Data
df_train_full = pd.read_parquet("df_final.parquet")
df_train_full = df_train_full[df_train_full["is_mule"] != -1].copy()

# Pre-process
X_full = build_features(df_train_full)
y_full = X_full["is_mule"].astype(int)
# Keep account_id out of training features but save it if needed
X_full = X_full.drop(columns=["is_mule", "account_id"], errors="ignore")

# Identify Column Types for Pipeline
num_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_full.select_dtypes(exclude=[np.number]).columns.tolist()

# Define Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols)
    ]
)

# Handle Class Imbalance
pos = int(y_full.sum())
neg = int(len(y_full) - pos)
scale_weight = neg / max(pos, 1)

model = LGBMClassifier(n_estimators=1000, learning_rate=0.03, scale_pos_weight=scale_weight, random_state=42)

pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", model)
])

# Split for Validation Scores
X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.2, stratify=y_full, random_state=42)

# Fit
pipeline.fit(X_train, y_train)

# Evaluate
val_probs = pipeline.predict_proba(X_val)[:, 1]
val_preds = pipeline.predict(X_val)

print("--- VALIDATION METRICS ---")
print(f"ROC AUC Score: {roc_auc_score(y_val, val_probs):.4f}")
print(f"F1 Score:      {f1_score(y_val, val_preds):.4f}")
print("\nClassification Report:\n", classification_report(y_val, val_preds))

[LightGBM] [Info] Number of positive: 2146, number of negative: 74726
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.049728 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19844
[LightGBM] [Info] Number of data points in the train set: 76872, number of used features: 134
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.027917 -> initscore=-3.550222
[LightGBM] [Info] Start training from score -3.550222


c:\Users\DHARMENDRA\Desktop\phase22\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\DHARMENDRA\Desktop\phase22\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


--- VALIDATION METRICS ---
ROC AUC Score: 0.9375
F1 Score:      0.7328

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99     18682
           1       0.68      0.79      0.73       537

    accuracy                           0.98     19219
   macro avg       0.84      0.89      0.86     19219
weighted avg       0.99      0.98      0.98     19219



In [ ]:
# 1. Load the real test file
df_test = pd.read_parquet("test_accounts.parquet")
test_ids = df_test["account_id"]

# 2. Process features exactly like training
X_test_final = build_features(df_test)

# 3. Ensure test columns match training columns exactly
X_test_final = X_test_final.reindex(columns=X_full.columns, fill_value=0)

# 4. Predict Probabilities (Hackathon requires 0 to 1 float)
final_probs = pipeline.predict_proba(X_test_final)[:, 1]

# 5. Create CSV
submission = pd.DataFrame({
    "account_id": test_ids,
    "is_mule": final_probs,
    "suspicious_start": "",
    "suspicious_end": ""
})

submission.to_csv("submission.csv", index=False)
print("Submission file 'submission.csv' generated!")
submission.head()